In [11]:
from evaluation.config import PROJECT_DIR
from evaluation.qa_datasets.eval_datasets import DATASETS
from evaluation.qa_datasets.google_client import GeminiApi
from dotenv import load_dotenv

load_dotenv()

gemini_api = GeminiApi()
base_dir = f'{PROJECT_DIR}/data'

### Gemini

In [7]:
answer_prompt_template = "Answer following query in at most five complete sentences: {question}"
model = 'gemini-3-pro-preview'

batch_names = []
for dataset_name, data in DATASETS.items():
    if dataset_name != 'truthful_qa':
        continue
    requests = []
    for sample in data['data']:
        request = {"key": sample['id'],
                   "request": {"contents": [{"parts": [{"text": answer_prompt_template.format(question=sample['question'])}], "role": "user"}],
                               "generationConfig": {"temperature": 0.0, "maxOutputTokens": 2048}}}
        requests.append(request)
    batch_name = gemini_api.create_batch(requests, f'{base_dir}/qa_datasets/{dataset_name}/{model}/batch-input-{model}-{dataset_name}.jsonl', model=model)
    output_file = f'{base_dir}/qa_datasets/{dataset_name}/gemini-3-pro-preview/batch-output-{model}-{dataset_name}.jsonl'
    batch_names.append((batch_name, output_file))

INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/upload/v1beta/files "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/upload/v1beta/files?upload_id=AGQBYWx9bSJdPhQF9CWVy_CH1UMPcaVsdZcd0hC85XjL1Vvq1L8GrCUjWdLwwVKJJM2nPcH2imcZykwuorRrjPycqtrSGzWBwzkLcPXfbRbn_2k&upload_protocol=resumable "HTTP/1.1 200 OK"


Uploaded file: files/u2h4fo9qyocv


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3-pro-preview:batchGenerateContent "HTTP/1.1 200 OK"


In [23]:
for name, file in batch_names:
    gemini_api.save_batch_results(name, file)

INFO:httpx:HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/batches/76bcrkeh5bn5pkl9izl2hhuuly9u2ib1ew5s "HTTP/1.1 200 OK"


Job state: JOB_STATE_RUNNING
